# **Part 6: Customer Vectors [Phase 2]**

As each customer has hundreds of transactions over the 6 months, Part 6 aims to compress each customer's entire purchase history into a single vector that captures their behavioural fingerprint.

*2 key versions* are built:
1. Weighted version (Primary method)
2. Simple mean (Baseline for comparison)

The *methodology* for Part 6 can be summarised as the following:
1. Load data and files from previous parts (Parts 1 - 5)
2. Build weighted vectors per customer: Frequency + Time Decay
3. Build baseliine vecotrs per customer

# ***To be deleted - as a reference for now***
**Why:** Each customer bought hundreds of different products over 6 months. We need to compress that history into one single profile vector per customer so we can mathematically compare customers to each other.

**Subtasks:**
- For each customer: look up the vector for every product they bought
- Average them — weighted by purchase frequency and recency (recent = higher weight)
- Add promo sensitivity as an extra feature (% of their purchases that were on promotion)
- Also compute a simple unweighted average as a baseline for comparison
- Save both versions

**Output:** `data/processed/customer_vectors_weighted.parquet` + `customer_vectors_baseline.parquet` — 1 row per customer

## 0. Check generated files from Parts 1-5 first

#### Check product_embeddings.parquet

In [6]:
import pandas as pd

emb = pd.read_parquet('../data/processed/product_embeddings.parquet')
print(emb.shape)
print(emb.dtypes)
print(emb.head(3))

(1000, 101)
idarticu      int64
v0          float64
v1          float64
v2          float64
v3          float64
             ...   
v95         float64
v96         float64
v97         float64
v98         float64
v99         float64
Length: 101, dtype: object
   idarticu        v0        v1        v2        v3        v4        v5  \
0         0  0.496714  1.399355 -0.675178 -1.907808 -0.863494 -0.423760   
1         1 -0.138264  0.924634 -0.144519 -0.860385 -0.031203 -0.453414   
2         2  0.647689  0.059630 -0.792420 -0.413606  0.018017 -1.795643   

         v6        v7        v8  ...       v90       v91       v92       v93  \
0 -1.114081  0.785185 -0.033025  ...  0.960895 -0.875781  0.038162  1.261745   
1 -0.630931 -1.777681 -0.503650  ... -0.369965  0.434668 -0.585079  0.007531   
2 -0.942060  0.714746 -0.172375  ... -0.579581 -0.193942  0.296729  2.066886   

        v94       v95       v96       v97       v98       v99  
0  0.193257 -0.508819  0.515058  1.742316 -0.301140  1.

#### Check df_combined to confirm columns

In [4]:
import polars as pl

df = pl.scan_parquet('../data/processed/df_combined.parquet')

for name, dtype in df.schema.items():
    print(f"{name}: {dtype}")

idempres: Int64
fecha: Date
hora: Int64
ticket: String
cliente: String
idarticu: Int64
unidades: Int64
importe: Float64
idpromoc: String
idtiprod: Int64
desc_larga_articulo: String
idsector: Int64
desc_sector: String


/var/folders/8g/cswm6ljd0h7fl4zd_8ndpblr0000gn/T/ipykernel_8576/3993894622.py:5: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  for name, dtype in df.schema.items():


## 1. Imports and Setups

In [7]:
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path

DATA_PROCESSED = Path("../data/processed")

print("Libraries loaded!")

Libraries loaded!


### 1.1 Load Product Embeddings
The output from Phase 1 (Word2Vec) is loaded first - Each of the 1000 products is now represented as a 100-number vector where products bought together sit geometrically close (e.g. beer will be located near chips, but far from diapers).

In [8]:
# load product embeddings
emb = pd.read_parquet(DATA_PROCESSED / "product_embeddings.parquet")

# identify the vector columns
vec_cols = [c for c in emb.columns if c.startswith("v")]

print(f"Embeddings shape: {emb.shape}")
print(f"Number of products: {len(emb):,}")
print(f"Vector size: {len(vec_cols)}")
print(f"\nSample:")
print(emb.head(3))

Embeddings shape: (1000, 101)
Number of products: 1,000
Vector size: 100

Sample:
   idarticu        v0        v1        v2        v3        v4        v5  \
0         0  0.496714  1.399355 -0.675178 -1.907808 -0.863494 -0.423760   
1         1 -0.138264  0.924634 -0.144519 -0.860385 -0.031203 -0.453414   
2         2  0.647689  0.059630 -0.792420 -0.413606  0.018017 -1.795643   

         v6        v7        v8  ...       v90       v91       v92       v93  \
0 -1.114081  0.785185 -0.033025  ...  0.960895 -0.875781  0.038162  1.261745   
1 -0.630931 -1.777681 -0.503650  ... -0.369965  0.434668 -0.585079  0.007531   
2 -0.942060  0.714746 -0.172375  ... -0.579581 -0.193942  0.296729  2.066886   

        v94       v95       v96       v97       v98       v99  
0  0.193257 -0.508819  0.515058  1.742316 -0.301140  1.356092  
1 -0.292425 -0.951266 -0.329052 -0.105780 -0.593205 -0.540008  
2  0.466237 -0.105022  1.327966 -0.069790  1.470130 -0.030894  

[3 rows x 101 columns]


### 1.2 Load Transaction Data
Only the 6 columns needed from the 190M rows of the cleaned dataset (df_combined.parquet) was loaded using streaming to avoid crashing the RAM. These columns include:
-`cliente` - customer ID
- `idarticu` - product ID (links to embeddings)
- `fetcha` - purchase date (needed for time decay)
- `unidades` - quantity (needed for frequency weighting)
- `importe` - price or amount
- `idpromoc` - promo flag (needed for promo sensitivity feature)

In [9]:
# load Transaction Data - uses streaming to avoid crashing RAM, memory safe
cols_needed = ["cliente", "idarticu", "fecha", "unidades", "importe", "idpromoc"]

print("Loading transactions (streaming)...")
df = (
    pl.scan_parquet(DATA_PROCESSED / "df_combined.parquet")
    .select(cols_needed)
    .collect(engine="streaming")
)

print(f"Transactions loaded: {len(df):,} rows")
print(f"Unique customers: {df['cliente'].n_unique():,}")
print(f"Unique products: {df['idarticu'].n_unique():,}")
print(df.head(3))

Loading transactions (streaming)...
Transactions loaded: 190,362,519 rows
Unique customers: 1,482,715
Unique products: 117,621
shape: (3, 6)
┌─────────────────────────────────┬──────────┬────────────┬──────────┬─────────┬──────────┐
│ cliente                         ┆ idarticu ┆ fecha      ┆ unidades ┆ importe ┆ idpromoc │
│ ---                             ┆ ---      ┆ ---        ┆ ---      ┆ ---     ┆ ---      │
│ str                             ┆ i64      ┆ date       ┆ i64      ┆ f64     ┆ str      │
╞═════════════════════════════════╪══════════╪════════════╪══════════╪═════════╪══════════╡
│ b95a4b1095b1ff396f5a52008f1663… ┆ 190776   ┆ 2022-06-13 ┆ 1        ┆ 3.97    ┆ No promo │
│ 4d6c59c00c7541352cfa2aafe7d567… ┆ 69819    ┆ 2022-04-16 ┆ 1        ┆ 3.1     ┆ No promo │
│ 6ea06a67a09b59087d6f991d9defce… ┆ 50203    ┆ 2022-04-17 ┆ 1        ┆ 1.66    ┆ No promo │
└─────────────────────────────────┴──────────┴────────────┴──────────┴─────────┴──────────┘


### 1.3 Filter to Products with Embeddings
Only transactions with products that have vectors are retained.

In [10]:
# Filter to products with embeddings & convert to pandas
products_with_embeddings = set(emb["idarticu"].values)
df_pd = df.filter(pl.col("idarticu").is_in(products_with_embeddings)).to_pandas()

print(f"Rows after filter: {len(df_pd):,}")
print(f"Unique customers: {df_pd['cliente'].nunique():,}")

Rows after filter: 2,112,705
Unique customers: 569,672


## 2. Time Decay
Time decay is computed with recent purchases weighted higher to reflect updated consumer purchasing behaviour and trends.

In [11]:
# Compute time decay — recent purchases weighted higher
REFERENCE_DATE = pd.Timestamp("2022-07-01")
DECAY_HALFLIFE_DAYS = 30

df_pd["days_ago"] = (REFERENCE_DATE - pd.to_datetime(df_pd["fecha"])).dt.days
df_pd["decay"] = np.exp(-np.log(2) * df_pd["days_ago"] / DECAY_HALFLIFE_DAYS)
df_pd["weight"] = df_pd["unidades"] * df_pd["decay"]

print("Time decay computed!")
print(df_pd[["fecha", "days_ago", "decay", "weight"]].head())

Time decay computed!
       fecha  days_ago     decay    weight
0 2022-04-12        80  0.157490  0.157490
1 2022-01-19       163  0.023142  0.185137
2 2022-05-15        47  0.337587  0.675175
3 2022-01-20       162  0.023683  0.023683
4 2022-04-03        89  0.127922  0.127922


## 3. Build Vectors
**Weighted Customer Vector** - This vector captures WHAT customers buy, HOW recently and HOW often\
For each customer:\
1. Look up the embedding vector for every product they bought
2. Multiply each vector by its weight (quantity x time decay)
3. Take the weighted average
4. Append promo sensitivty (% of purchases on promotion)

**Baseline Customer Vector**\
Similar process but uses a simple unweighted mean and does not have a frequency or recency weighting. This is the comparison baseline to validate whether the weighted aggregation produces better clusters.

In [12]:
# Map product to embedding vector
emb_matrix = emb[vec_cols].values.astype(np.float32)
emb_index = {pid: i for i, pid in enumerate(emb["idarticu"])}
df_pd["emb_idx"] = df_pd["idarticu"].map(emb_index)

In [13]:
# Promo sensitivity per customer
promo_rate = df_pd.groupby("cliente")["idpromoc"].apply(
    lambda x: (x == "Promo").mean()
).rename("promo_sensitivity")

In [14]:
# Build vectors
print("Building customer vectors...")
weighted_rows = []
baseline_rows = []

for cust_id, grp in df_pd.groupby("cliente"):
    vecs = emb_matrix[grp["emb_idx"].values]
    w = grp["weight"].values
    
    # Weighted
    weighted = (vecs * w[:, None]).sum(axis=0) / (w.sum() + 1e-9)
    # Baseline
    baseline = vecs.mean(axis=0)
    
    weighted_rows.append([cust_id] + weighted.tolist())
    baseline_rows.append([cust_id] + baseline.tolist())

print(f"Done! Built vectors for {len(weighted_rows):,} customers")

Building customer vectors...
Done! Built vectors for 569,672 customers


In [15]:
# Save both outputs
cols_out = ["cliente"] + vec_cols

df_weighted = pd.DataFrame(weighted_rows, columns=cols_out)
df_weighted["promo_sensitivity"] = df_weighted["cliente"].map(promo_rate)
df_weighted.to_parquet(DATA_PROCESSED / "customer_vectors_weighted.parquet", index=False)

df_baseline = pd.DataFrame(baseline_rows, columns=cols_out)
df_baseline.to_parquet(DATA_PROCESSED / "customer_vectors_baseline.parquet", index=False)

print(f"Saved weighted: {df_weighted.shape}")
print(f"Saved baseline: {df_baseline.shape}")

Saved weighted: (569672, 102)
Saved baseline: (569672, 101)


In [16]:
# Sanity check — verify outputs
w = pd.read_parquet(DATA_PROCESSED / "customer_vectors_weighted.parquet")
b = pd.read_parquet(DATA_PROCESSED / "customer_vectors_baseline.parquet")
print(f"Weighted: {w.shape} ✅")
print(f"Baseline: {b.shape} ✅")
print(f"Sample weighted vector:\n{w.head(2)}")

Weighted: (569672, 102) ✅
Baseline: (569672, 101) ✅
Sample weighted vector:
                                             cliente        v0        v1  \
0  00004724b4bb68fdb2737bdd3f3864f0b0cb1ad008e79e...  0.346448 -2.153390   
1  00009bb0efdb197afc2a82f515e0f1bf6c491ae5849cfd... -1.478522 -0.172627   

        v2        v3        v4        v5        v6        v7        v8  ...  \
0  1.66147 -0.369207  1.127780 -0.852225 -0.883719  0.741074 -0.544296  ...   
1 -0.82242  0.714610  0.932435  0.828059  0.109572  0.797975  0.953176  ...   

        v91       v92       v93       v94       v95       v96       v97  \
0  0.323069 -1.051290  2.032660 -0.704094 -0.258499  0.212244 -1.010208   
1  0.787001  3.594374  1.505267  1.088649  0.662524 -0.432046  1.901494   

        v98       v99  promo_sensitivity  
0 -1.539832 -0.449030               0.25  
1  0.487645 -0.073648               0.00  

[2 rows x 102 columns]
